# 08 — Analyses croisées

## Objectif

Cette analyse croise plusieurs variables du dataset afin d'identifier des relations et des tendances entre les contenus.

Nous allons notamment étudier les relations entre :

- le type de contenu et l'année de sortie ;
- le type de contenu et la classification ;
- les catégories et le type de contenu ;
- les pays et le type de contenu ;
- l'année de sortie et la classification.


## 1. Importation des bibliothèques

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


## 2. Chargement des données

In [ ]:
df = pd.read_csv("../data/processed/netflix_titles_clean.csv")

df["date_added"] = pd.to_datetime(
    df["date_added"],
    errors="coerce"
)

print(f"Nombre de contenus : {len(df)}")


## 3. Type de contenu × année de sortie

In [ ]:
type_year = (
    df.groupby(["release_year", "type"])
    .size()
    .unstack(fill_value=0)
)

type_year.tail(20)


In [ ]:
plt.figure(figsize=(12, 6))

for content_type in type_year.columns:
    plt.plot(
        type_year.index,
        type_year[content_type],
        label=content_type
    )

plt.title("Évolution des films et séries selon l'année de sortie")
plt.xlabel("Année de sortie")
plt.ylabel("Nombre de contenus")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Type de contenu × classification

In [ ]:
rating_type = pd.crosstab(
    df["rating"],
    df["type"]
)

rating_type


In [ ]:
rating_type.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Classification selon le type de contenu")
plt.xlabel("Classification")
plt.ylabel("Nombre de contenus")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Catégories × type de contenu

In [ ]:
df_categories = df.assign(
    category=df["listed_in"].str.split(", ")
).explode("category")

category_type = (
    df_categories
    .groupby(["category", "type"])
    .size()
    .unstack(fill_value=0)
)

category_type["Total"] = category_type.sum(axis=1)

category_type.sort_values(
    "Total",
    ascending=False
).head(15)


## 6. Pays × type de contenu

In [ ]:
df_countries = df.assign(
    country=df["country"].str.split(", ")
).explode("country")

country_type = (
    df_countries
    .groupby(["country", "type"])
    .size()
    .unstack(fill_value=0)
)

country_type["Total"] = country_type.sum(axis=1)

country_type.sort_values(
    "Total",
    ascending=False
).head(15)


## 7. Année × classification

In [ ]:
recent_df = df[
    df["release_year"] >= 2010
].copy()

year_rating = pd.crosstab(
    recent_df["release_year"],
    recent_df["rating"]
)

year_rating.tail(15)


## 8. Évolution des principales classifications

In [ ]:
top_ratings = (
    df["rating"]
    .value_counts()
    .head(5)
    .index
)

rating_evolution = (
    df[df["rating"].isin(top_ratings)]
    .groupby(["release_year", "rating"])
    .size()
    .unstack(fill_value=0)
)

rating_evolution.tail(20)


In [ ]:
plt.figure(figsize=(12, 6))

for rating in rating_evolution.columns:
    plt.plot(
        rating_evolution.index,
        rating_evolution[rating],
        label=rating
    )

plt.title("Évolution des principales classifications")
plt.xlabel("Année de sortie")
plt.ylabel("Nombre de contenus")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Nombre de catégories selon le type

In [ ]:
df["category_count"] = (
    df["listed_in"]
    .str.split(", ")
    .str.len()
)

category_count_by_type = (
    df.groupby("type")["category_count"]
    .agg(["mean", "median", "max"])
    .round(2)
)

category_count_by_type


## 10. Nombre de pays associés selon le type

In [ ]:
df["country_count"] = (
    df["country"]
    .str.split(", ")
    .str.len()
)

country_count_by_type = (
    df.groupby("type")["country_count"]
    .agg(["mean", "median", "max"])
    .round(2)
)

country_count_by_type


## 11. Contenus récents × type

In [ ]:
recent_content = df[
    df["release_year"] >= 2015
]

recent_distribution = (
    recent_content["type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

recent_distribution


In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=recent_content,
    x="type"
)

plt.title("Répartition des contenus sortis depuis 2015")
plt.xlabel("Type")
plt.ylabel("Nombre de contenus")
plt.tight_layout()
plt.show()


## 12. Principaux résultats

In [ ]:
print("=== TYPE ===")
print(df["type"].value_counts())

print("\n=== PRINCIPALES CLASSIFICATIONS ===")
print(df["rating"].value_counts().head(5))

print("\n=== ANNÉES LES PLUS REPRÉSENTÉES ===")
print(df["release_year"].value_counts().head(5))

print("\n=== CATÉGORIES PRINCIPALES ===")
print(df_categories["category"].value_counts().head(5))

print("\n=== PAYS PRINCIPAUX ===")
print(df_countries["country"].value_counts().head(5))


## 13. Conclusion

Les analyses croisées permettent d'aller au-delà des distributions individuelles.

Elles mettent en relation plusieurs caractéristiques du catalogue et permettent d'identifier des tendances selon le type de contenu, la période, les classifications, les catégories et les pays.

Ces résultats serviront de base à la synthèse finale du projet.
